# Classifiers, feature selection and tuning

**Purpose:** Compare classifiers on shared development folds, record independent feature selection, and choose a model without test outcomes.

Requires the complete project folder and dependencies from `requirements.txt`; see README.md for local/Colab/Kaggle setup. Matching saved artifacts are reused. Use `force=True` in the phase cell to recompute. Outputs must remain visible when saving.

In [1]:
from pathlib import Path
import os, sys
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ.setdefault(name, '2')
candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/hospital-readmission'), Path('/kaggle/working/hospital-readmission')]
ROOT = next((p for p in candidates if (p / 'config.json').exists() and (p / 'readmission').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the full project and set the working directory to hospital-readmission first. See README.md.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, JSON
from readmission.common import read_json
from readmission.pipeline import run_phase
pd.set_option('display.max_columns', 30)
print('Project:', ROOT.name)


Project: hospital-readmission


In [2]:
run_phase('models')

Phase data: reusing verified matching artifacts.


Phase models: reusing verified matching artifacts.


In [3]:
display(pd.read_csv(ROOT / 'artifacts/tables/cv_split_audit.csv'))

,fold,training_encounters,validation_encounters,training_positive_rate,validation_positive_rate,patient_overlap
0,0,47610,12060,0.115102,0.113930,0
1,1,47769,11901,0.114782,0.115200,0
2,2,47746,11924,0.113455,0.120513,0
3,3,47759,11911,0.115978,0.110402,0
4,4,47796,11874,0.115010,0.114283,0


In [4]:
display(pd.read_csv(ROOT / 'artifacts/tables/model_selection.csv'))

,experiment,family,feature_set,baseline_cv_ap,selected_cv_ap,sampled_best_cv_ap,cv_ap_gain,selected_candidate,raw_inputs_retained,encoded_inputs,fit_seconds,search_seconds,best_params,warnings
0,logistic_regression__all,logistic_regression,all,0.217503,0.217503,0.217124,0.000000,baseline,42,216,5.034353,223.239822,{},NaN
1,logistic_regression__mi20,logistic_regression,mi20,0.218184,0.218184,0.218022,0.000000,baseline,20,147,3.283494,146.760867,{},NaN
2,random_forest__all,random_forest,all,0.225676,0.225676,0.224929,0.000000,baseline,42,216,13.912290,367.986181,{},NaN
3,random_forest__mi20,random_forest,mi20,0.226428,0.226428,0.224891,0.000000,baseline,20,147,10.028791,293.897779,{},NaN
4,xgboost__all,xgboost,all,0.225769,0.231683,0.231683,0.005914,random_search,42,216,3.633880,148.875945,"{""model__subsample"": 1.0, ""model__min_child_we...",NaN
5,xgboost__mi20,xgboost,mi20,0.227321,0.231491,0.231491,0.004170,random_search,20,147,2.787660,116.108922,"{""model__subsample"": 1.0, ""model__min_child_we...",NaN


In [5]:
display(pd.read_csv(ROOT / 'artifacts/tables/cv_fold_scores.csv'))

,experiment,fold,stage,average_precision,fit_seconds
0,logistic_regression__all,0,baseline,0.213242,10.773077
1,logistic_regression__all,0,selected,0.213242,10.773077
2,logistic_regression__all,1,baseline,0.214006,10.944810
3,logistic_regression__all,1,selected,0.214006,10.944810
4,logistic_regression__all,2,baseline,0.226067,10.868554
5,logistic_regression__all,2,selected,0.226067,10.868554
6,logistic_regression__all,3,baseline,0.219902,10.744352
7,logistic_regression__all,3,selected,0.219902,10.744352
8,logistic_regression__all,4,baseline,0.214297,10.400172
9,logistic_regression__all,4,selected,0.214297,10.400172


In [6]:
features = pd.read_csv(ROOT / 'artifacts/tables/feature_selection.csv')
display(features.query("experiment == 'logistic_regression__mi20'").sort_values('mutual_information', ascending=False))

,experiment,feature,mutual_information,selected
56,logistic_regression__mi20,number_inpatient,0.011475,True
46,logistic_regression__mi20,discharge_disposition_id,0.005763,True
55,logistic_regression__mi20,number_emergency,0.002387,True
50,logistic_regression__mi20,medical_specialty,0.001957,True
53,logistic_regression__mi20,num_medications,0.001875,True
48,logistic_regression__mi20,time_in_hospital,0.001663,True
60,logistic_regression__mi20,number_diagnoses,0.001662,True
51,logistic_regression__mi20,num_lab_procedures,0.001379,True
54,logistic_regression__mi20,number_outpatient,0.000982,True
77,logistic_regression__mi20,insulin,0.000820,True


In [7]:
display(JSON(read_json('artifacts/model_choice.json')))

<IPython.core.display.JSON object>

In [8]:
for filename in sorted((ROOT / 'artifacts/tables').glob('search_*.csv')):
    print(filename.name)
    display(pd.read_csv(filename)[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'rank_test_score']])

search_logistic_regression__all.csv


,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,"{'model__class_weight': 'balanced', 'model__C'...",0.216687,0.005060,9.127238,4
1,"{'model__class_weight': 'balanced', 'model__C'...",0.216909,0.005138,8.188379,3
2,"{'model__class_weight': None, 'model__C': 10}",0.217124,0.004583,8.198430,1
3,"{'model__class_weight': 'balanced', 'model__C'...",0.217070,0.005495,6.851366,2


search_logistic_regression__mi20.csv


,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,"{'model__class_weight': 'balanced', 'model__C'...",0.217525,0.006184,5.897495,3
1,"{'model__class_weight': 'balanced', 'model__C'...",0.217573,0.006194,5.259537,2
2,"{'model__class_weight': None, 'model__C': 10}",0.218022,0.005663,5.354951,1
3,"{'model__class_weight': 'balanced', 'model__C'...",0.217477,0.006237,5.699270,4


search_random_forest__all.csv


...

,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,"{'model__min_samples_leaf': 5, 'model__max_dep...",0.215122,0.008629,11.858117,2
1,"{'model__min_samples_leaf': 15, 'model__max_de...",0.224891,0.008721,11.754670,1
2,"{'model__min_samples_leaf': 30, 'model__max_de...",0.211001,0.005881,7.445984,4
3,"{'model__min_samples_leaf': 30, 'model__max_de...",0.214050,0.007000,15.022732,3


search_xgboost__all.csv


,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,"{'model__subsample': 1.0, 'model__min_child_we...",0.218794,0.004340,5.384133,3
1,"{'model__subsample': 0.8, 'model__min_child_we...",0.223064,0.005192,5.529955,2
2,"{'model__subsample': 1.0, 'model__min_child_we...",0.218586,0.005105,5.320966,4
3,"{'model__subsample': 1.0, 'model__min_child_we...",0.231683,0.003936,5.626526,1


search_xgboost__mi20.csv


,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,"{'model__subsample': 1.0, 'model__min_child_we...",0.218863,0.004368,4.301557,3
1,"{'model__subsample': 0.8, 'model__min_child_we...",0.223432,0.005626,4.498441,2
2,"{'model__subsample': 1.0, 'model__min_child_we...",0.218609,0.005290,4.228236,4
3,"{'model__subsample': 1.0, 'model__min_child_we...",0.231491,0.005559,4.461298,1


## Team interpretation prompts

Explain the tuning search, baseline retention rule, selected raw/encoded features, and why held-out test results must not choose the model.

Write your own report discussion after checking the evidence.